**数据预处理：pandas**<a id='toc0_'></a>    
- [读取数据集](#toc1_)    
- [处理缺失值](#toc2_)    
- [转换为张量格式](#toc3_)    
- [小结](#toc4_)    
- [练习](#toc5_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

<div style="font-size:32px;  padding:5px;"> 
数据预处理：pandas
</div>

为了能用深度学习来解决现实世界的问题，我们经常从预处理原始数据开始，
而不是从那些准备好的张量格式数据开始。
在Python中常用的数据分析工具中，我们通常使用[pandas库](https://pandas.pydata.org/docs/user_guide/index.html)软件包。
像庞大的Python生态系统中的许多其他扩展包一样，`pandas`可以与张量兼容。
本节内容虽然不能替代一份系统的 *pandas*
[教程](https://pandas.pydata.org/pandas-docs/stable/user_guide/10min.html)，
但将为你提供一次速成入门，
介绍一些最常见的操作流程。
后面的章节将介绍更多的数据预处理技术。

⚠️ `pandas`中`fillna()`等数据处理方法默认是非原地（non-inplace）的

$\implies$ 它们不会直接修改原始`DataFrame/Series`，而是返回一个修改后的新副本。

# <a id='toc1_'></a>[读取数据集](#toc0_)

逗号分隔值（CSV）文件在存储表格型（类似电子表格）数据时极为常见。
在这种文件中，每一行对应一条记录，
并由若干个（以逗号分隔的）字段组成，例如：
> “Albert Einstein,March 14 1879,Ulm,Federal polytechnic school,field of gravitational physics”.

为了演示如何使用 `pandas` 加载 CSV 文件，我们首先**创建一个人工数据集，并存储在CSV（逗号分隔值）文件**
`../data/house_tiny.csv`中。

该文件表示一个房屋数据集，其中:
* 每一行对应一套不同的房屋，
* 各列分别表示房间数量（`NumRooms`）、屋顶类型（`RoofType`）以及价格（`Price`）。

以其他格式存储的数据也可以通过类似的方式进行处理。

In [ ]:
import os
# 手搓数据集
os.makedirs(os.path.join('..', 'data'), exist_ok=True)# 如果该文件夹存在的话，也不报错而是继续运行
data_file = os.path.join('..', 'data', 'house_tiny.csv')# 如果这csv存在，也不会报错，因为只是拼接路径
with open(data_file, 'w') as f:# 如果不存在就创建+写入，已存在就清空+写入，不想清空就用a
    f.write('''NumRooms,RoofType,Price
NA,NA,127500
2,NA,106000
4,Slate,178100
NA,NA,140000''')#第一行是列名

【数据读取】操作如下。我们导入`pandas`包并调用`read_csv`函数。该数据集有四行三列。

In [1]:
# 如果没有安装pandas，只需取消对以下行的注释来安装pandas
# !pip install pandas
import os
import pandas as pd
data_file = os.path.join('..', 'data', 'house_tiny.csv')
data = pd.read_csv(data_file)
print(data)

   NumRooms RoofType   Price
0       NaN      NaN  127500
1       2.0      NaN  106000
2       4.0    Slate  178100
3       NaN      NaN  140000


# <a id='toc2_'></a>[缺失值处理](#toc0_)

在监督学习中，我们训练模型在给定一组*输入*值的情况下，去预测一个指定的*目标*值。

处理数据集的第一步是将对应于输入值和目标值的列分离开来。

🔔 我们既可以按列名选择列，也可以通过基于【整数位置】的索引（`iloc`）来选择。

In [ ]:
inputs, targets = data.iloc[:, 0:2], data.iloc[:, 2]# 通过位置索引iloc，我们将data分成inputs和outputs

print("inputs:")# DataFrame (二维表格)
print(inputs)

print("targets:")# Series (一维数组)
print(targets)# 当你使用整数索引 iloc[:, 2] 取出单列时，Pandas 会默认将数据“降维”。它不再是一个表格，而变成了一个带有标签的一维数组
# 在打印一维数组的时候，是先打印数据（索引和值），再在底部打印该序列的元数据（metadata），包括那么name:原本列名，dtype：数据类型

inputs:
   NumRooms RoofType
0       NaN      NaN
1       2.0      NaN
2       4.0    Slate
3       NaN      NaN
targets:
0    127500
1    106000
2    178100
3    140000
Name: Price, dtype: int64


In [ ]:
# 补充

你可能已经注意到，`pandas` 会将CSV 中所有取值为 `NA` 的条目
替换为一种特殊的 `NaN`（*not a number*而不是例如9999这样的具体的数）值。
当某个条目为空时也会发生这种情况，例如 `"3,,,270000"`。
这些被称为*缺失值*。它们是数据科学中的“臭虫”，一种在你整个职业生涯中都会反复遇到的顽固问题。


⚠️ 常见处理方法如下，请根据具体情况使用：

* 【**插值法mputation**】：这个缺失值的估计值是啥，再进行替换。
* 【**删除法deletion**】：直接丢掉包含缺失值的行或者列。


在这里我们考虑插值法，下面是一些常见的启发式插值法：

🌸 **对于分类型输入字段，我们可以将 `NaN` 视为一个类别。**

由于 `RoofType` 列的取值为 `Slate` 和 `NaN`，因此`pandas` 可以将这一列
转换为两列：`RoofType_Slate` 和 `RoofType_nan`。
如果某一行的屋顶类型是 `Slate`，
那么 `RoofType_Slate` 和 `RoofType_nan`
的取值分别为 1 和 0。
反之，对于屋顶类型缺失的那一行，
取值成了 0 和 1。


🌸 **对于数值型数据，缺失值可以用对应列的平均值来替换这些 `NaN` 项。**

In [ ]:

inputs = pd.get_dummies(inputs, dummy_na=True)# # 对inputs中的分类特征执行独热编码（将离散类别转为数值型特征）  
# dummy_na=True → 特殊处理：将缺失值(NA)视为一个独立的类别，生成对应的编码列  
# 例如：若某列有"NA"值，会新增一列如"列名_na"来标识该缺失情况 
print(inputs)

In [ ]:
inputs = inputs.fillna(inputs.mean())
print(inputs)

In [ ]:
# 当然也可以
data.iloc[:, 0] = data.iloc[:,0].fillna(data.iloc[:,0].mean())# 需要在下一句之前
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]# 分开并起名
print(inputs)

⚠️  `pandas`使用`[]`会被误解成按列名筛选，因此需要使用位置索引`iloc`或者标签索引`loc`来选择

In [ ]:
inputs['NumRooms'] = inputs['NumRooms'].fillna(inputs['NumRooms'].mean())# 第三种办法：直接使用列名
print(inputs)

In [ ]:
#import numpy as np
#inputs['Alley'] = inputs['Alley'].replace('NaN', np.nan)
inputs = pd.get_dummies(inputs, dummy_na=True, dtype=int)#不设置为int就出现的是false或者true
print(inputs)

# <a id='toc3_'></a>[转换为张量格式](#toc0_)

现在`inputs`和`outputs`中的所有条目都是【**数值类型** 】

$\implies$ 可以转换为【**张量格式**】。

当数据采用张量格式后，可以通过在[ndarray](ndarray.ipynb)中引入的那些张量函数来进一步操作。


In [ ]:
import torch

X1 = torch.tensor(inputs.to_numpy(dtype=float))
y1 = torch.tensor(outputs.to_numpy(dtype=float))
X1, y1

In [ ]:
X2,y2 = torch.tensor(inputs.values),torch.tensor(outputs.values)
X2,y2

# <a id='toc4_'></a>[小结](#toc0_)

* `pandas`软件包是Python中常用的数据分析工具中，`pandas`可以与张量兼容。
* 用`pandas`处理缺失的数据时，我们可根据情况选择用插值法和删除法。

# <a id='toc5_'></a>[练习](#toc0_)

创建包含更多行和列的原始数据集。

1. 删除缺失值最多的列。
2. 将预处理后的数据集转换为张量格式。


In [ ]:
import os
import pandas as pd
data_file = os.path.join('..', 'data', 'house_tiny.csv')
data = pd.read_csv(data_file)


In [ ]:
# 删除缺失值最多的列
null_count = data.iloc[:,0:2].isnull().sum()
max_null = null_count.idxmax()
data.drop(max_null, axis = 1, inplace=True)
print(data)

In [ ]:
data_tensor = torch.tensor(data.values)
print(data_tensor)